# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hafsa-SE/flyrank-assignment/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two signal checks, then my rule

**Signal check A — staleness (behind the refresh flags).** Claim: "pages that haven't been
updated in a long time are more likely to be declining." I bucket by `freshness_tier`
(from `days_since_last_update`) and look at the decline rate (`trend_direction == 'down'`)
in each bucket, with `n` printed so no bucket under the ~50-row floor gets a verdict.

**Signal check B — CTR vs. position (behind the CTR-fix logic).** Claim: "pages ranking
better (lower `avg_position`) get a higher click-through rate." I bucket by `position_tier`
and compare the **impression-weighted** CTR (total clicks / total impressions per bucket —
not the mean of per-row CTRs, which over-weights tiny-volume pages) across tiers, restricted
to rows with real position data (`avg_position > 0`).

**My rule, in plain words:** "A page is worth reviewing for a CTR fix if it is already
visible (real search demand), sitting in a decent ranking position, but its actual CTR is
below what other pages *in that same position tier* achieve — the gap between what the spot
should earn and what it's actually earning, weighted by how much demand is on the table."

This deliberately does **not** use `trend_direction` or `trend_pct` as inputs — those define
the label (`is_declining_label`) per the data dictionary, so they're evaluation-only, never
features.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 120)

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)  # EVALUATION ONLY, never a feature
print(f"Loaded {len(df):,} rows, {df['client_id'].nunique()} clients")

# ---------------------------------------------------------------
# Signal check A: staleness vs decline rate (behind refresh flags)
# ---------------------------------------------------------------
sig_a = (
    df.groupby("freshness_tier", observed=True)
      .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
      .reset_index()
      .sort_values("decline_rate", ascending=False)
)
print("\n--- Signal A: freshness_tier vs decline rate ---")
print(sig_a.to_string(index=False))

FLOOR = 50
sig_a_valid = sig_a[sig_a["n"] >= FLOOR]
spread_a = sig_a_valid["decline_rate"].max() - sig_a_valid["decline_rate"].min()
print(f"\nBuckets below the n>={FLOOR} floor are excluded from the verdict.")
print(f"Decline-rate spread across valid buckets: {spread_a:.3f}")
print("VERDICT (Signal A — staleness): MIXED")
print("Explanation: decline rate is NOT monotonic in staleness. The freshest bucket (0-30d, n=20,480)")
print("already has a ~51% decline rate, and the most-stale bucket (181+, n=174) has the LOWEST decline")
print("rate (~47%) of any tier -- the opposite of the 'old pages decline more' story. The 91-180d tier")
print("is highest (~61%) but that's the middle of the range, not the tail. Staleness alone is not a")
print("clean predictor of decline here -- worth knowing before leaning on it as the sole trigger.")

# ---------------------------------------------------------------
# Signal check B: CTR vs position (behind the CTR-fix logic)
# ---------------------------------------------------------------
sub = df[df["avg_position"] > 0].copy()  # avg_position == 0 means "no data", not rank zero
order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

def weighted_ctr(g):
    return pd.Series({
        "n": len(g),
        "weighted_ctr_pct": 100 * g["clicks_90d"].sum() / g["impressions_90d"].sum() if g["impressions_90d"].sum() else 0,
        "mean_row_ctr_pct": g["ctr"].mean(),
        "median_impressions_90d": g["impressions_90d"].median(),
    })

sig_b = sub.groupby("position_tier", observed=True).apply(weighted_ctr, include_groups=False).reset_index()
sig_b["position_tier"] = pd.Categorical(sig_b["position_tier"], categories=order, ordered=True)
sig_b = sig_b.sort_values("position_tier")
print("\n--- Signal B: position_tier vs CTR (weighted) ---")
print(sig_b.to_string(index=False))

sig_b_valid = sig_b[sig_b["n"] >= FLOOR]
is_monotonic = sig_b_valid["weighted_ctr_pct"].is_monotonic_decreasing
print(f"\nAll buckets clear the n>={FLOOR} floor.")
print(f"Weighted CTR strictly decreases as position gets worse (top_3 -> deep): {is_monotonic}")
print("VERDICT (Signal B — CTR vs position): CONFIRMED")
print("Explanation: weighted CTR falls monotonically from top_3 (0.49%) down to deep (0.04%) --")
print("better position genuinely earns more clicks per impression here. Note the gap between")
print("weighted CTR and the mean-of-row-CTRs column for top_3 (0.49% vs 2.76%): that tier's median")
print("volume is only ~53 impressions/90d, so a couple of high-CTR/low-volume pages skew the simple")
print("average -- the data dictionary's warning about needing a volume floor on position_tier is real,")
print("which is exactly why I weight by impressions instead of averaging per-row rates.")


Loaded 30,000 rows, 32 clients

--- Signal A: freshness_tier vs decline rate ---
freshness_tier     n  decline_rate
        91-180  9171      0.611057
         31-90   175      0.588571
          0-30 20480      0.511377
          181+   174      0.471264

Buckets below the n>=50 floor are excluded from the verdict.
Decline-rate spread across valid buckets: 0.140
VERDICT (Signal A — staleness): MIXED
Explanation: decline rate is NOT monotonic in staleness. The freshest bucket (0-30d, n=20,480)
already has a ~51% decline rate, and the most-stale bucket (181+, n=174) has the LOWEST decline
rate (~47%) of any tier -- the opposite of the 'old pages decline more' story. The 91-180d tier
is highest (~61%) but that's the middle of the range, not the tail. Staleness alone is not a
clean predictor of decline here -- worth knowing before leaning on it as the sole trigger.

--- Signal B: position_tier vs CTR (weighted) ---
position_tier       n  weighted_ctr_pct  mean_row_ctr_pct  median_impressi

## 2. Build the ranked queue (writes the CSV)

Rule, coded as a transparent score (no fitted weights): visible pages, in a real position
tier, scored by how far their CTR sits below the tier's own median CTR, weighted by how much
search demand (impressions) is actually on the table. One reason code marks every row the
rule flags; everything else gets no flag and no action.


In [2]:
VISIBILITY_FLOOR = 500  # impressions_90d -- matches the 'visible page' bar used in the session

measurable = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= VISIBILITY_FLOOR)].copy()
tier_benchmark_ctr = measurable.groupby("position_tier", observed=True)["ctr"].median()
print("Tier benchmark CTR (median, visible-only slice):")
print(tier_benchmark_ctr)

work = df.copy()
work["tier_benchmark_ctr"] = work["position_tier"].map(tier_benchmark_ctr)
work["is_visible"] = (work["avg_position"] > 0) & (work["impressions_90d"] >= VISIBILITY_FLOOR)

# ctr_gap: how far below the tier benchmark this page's CTR sits (0 if at/above benchmark, or not visible)
ctr_gap = (work["tier_benchmark_ctr"] - work["ctr"]).clip(lower=0)
work["ctr_gap"] = np.where(work["is_visible"], ctr_gap, 0.0)

# score: readable on purpose -- gap in CTR points * log-scaled demand, zero for non-visible pages
work["baseline_action_score"] = work["ctr_gap"] * np.log1p(work["impressions_90d"])
work["is_flagged"] = work["baseline_action_score"] > 0

REASON_CODE = "ctr_underperforming_visible_page"   # the ONE reason code this rule emits
ACTION_FLAGGED = "fix_ctr_snippet_and_meta"
ACTION_DEFAULT = "monitor"

work["reason_code"] = np.where(work["is_flagged"], REASON_CODE, "")
work["action"] = np.where(work["is_flagged"], ACTION_FLAGGED, ACTION_DEFAULT)

work["baseline_rank"] = (
    work["baseline_action_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

output_cols = [
    "baseline_rank", "content_id", "client_id",
    "baseline_action_score", "reason_code", "action",
    "position_tier", "avg_position", "ctr", "tier_benchmark_ctr", "ctr_gap",
    "impressions_90d", "days_since_last_update", "freshness_tier",
    "is_declining_label",  # kept for evaluation only -- NOT used to build the score above
]
out = work[output_cols].sort_values("baseline_rank").reset_index(drop=True)

out_path = Path("../outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(out_path, index=False)
print(f"\nWrote {len(out):,} rows to {out_path}")
print(f"Flagged rows: {int(work['is_flagged'].sum()):,} ({work['is_flagged'].mean():.1%} of all rows)")

# Honest evaluation: precision@K against the label, base rate alongside it
def precision_at_k(frame, k, label_col="is_declining_label"):
    top = frame.head(k)
    return top[label_col].mean() if len(top) else float("nan")

base_rate = out["is_declining_label"].mean()
for k in (10, 20, 50):
    print(f"Precision@{k}: {precision_at_k(out, k):.3f}   (base rate: {base_rate:.3f}, n={len(out)})")

out.head(10)


Tier benchmark CTR (median, visible-only slice):
position_tier
deep        0.00
page_1      0.24
page_3_5    0.09
striking    0.17
top_3       0.20
Name: ctr, dtype: float64



Wrote 30,000 rows to ../outputs/baseline_action_score.csv
Flagged rows: 7,950 (26.5% of all rows)
Precision@10: 0.700   (base rate: 0.542, n=30000)
Precision@20: 0.700   (base rate: 0.542, n=30000)
Precision@50: 0.680   (base rate: 0.542, n=30000)


,baseline_rank,content_id,client_id,baseline_action_score,reason_code,action,position_tier,avg_position,ctr,tier_benchmark_ctr,ctr_gap,impressions_90d,days_since_last_update,freshness_tier,is_declining_label
0,1,content_c8e9d6ab9013,client_19581e27de,2.939653,ctr_underperforming_visible_page,fix_ctr_snippet_and_meta,page_1,9.7,0.00,0.24,0.24,208678,104,91-180,1
1,2,content_453722754fea,client_f369cb89fc,2.725493,ctr_underperforming_visible_page,fix_ctr_snippet_and_meta,page_1,7.6,0.01,0.24,0.23,140079,20,0-30,1
2,3,content_39881853ef0c,client_f369cb89fc,2.674930,ctr_underperforming_visible_page,fix_ctr_snippet_and_meta,page_1,7.2,0.01,0.24,0.23,112434,20,0-30,1
3,4,content_c84a0ab98e90,client_f369cb89fc,2.586391,ctr_underperforming_visible_page,fix_ctr_snippet_and_meta,page_1,7.8,0.03,0.24,0.21,223271,20,0-30,0
4,5,content_0919dd345d80,client_4e07408562,2.571516,ctr_underperforming_visible_page,fix_ctr_snippet_and_meta,page_1,7.0,0.02,0.24,0.22,119217,7,0-30,1
5,6,content_d274ac4158ef,client_4e07408562,2.549384,ctr_underperforming_visible_page,fix_ctr_snippet_and_meta,page_1,6.8,0.01,0.24,0.23,65138,26,0-30,0
6,7,content_e5f459e737b7,client_f369cb89fc,2.516105,ctr_underperforming_visible_page,fix_ctr_snippet_and_meta,page_1,5.9,0.01,0.24,0.23,56363,20,0-30,1
7,8,content_c1fe78bc4e37,client_19581e27de,2.479263,ctr_underperforming_visible_page,fix_ctr_snippet_and_meta,page_1,7.5,0.03,0.24,0.21,134055,104,91-180,1
8,9,content_339b357d04c7,client_bbb965ab0c,2.473730,ctr_underperforming_visible_page,fix_ctr_snippet_and_meta,page_1,3.7,0.01,0.24,0.23,46879,15,0-30,0
9,10,content_65114d89496d,client_19581e27de,2.462495,ctr_underperforming_visible_page,fix_ctr_snippet_and_meta,page_1,6.5,0.02,0.24,0.22,72631,22,0-30,1


## 3. Top-10 review

Assignment card asks for a top-10 (the skeleton title says top-20 -- top-10 is what's
required; a top-20 is an optional deeper pass, not attempted here). One line each: the
action, why it's there, and what would make it wrong.


In [3]:
top10 = out.head(10).copy()

for i, row in top10.iterrows():
    print(f"#{row['baseline_rank']:>2}  content_id={row['content_id']}")
    print(f"    action: {row['action']}  |  reason: {row['reason_code']}")
    print(
        f"    why: position_tier={row['position_tier']} (avg_position={row['avg_position']:.1f}), "
        f"ctr={row['ctr']:.2f}% vs tier benchmark {row['tier_benchmark_ctr']:.2f}% "
        f"(gap={row['ctr_gap']:.2f}pp), impressions_90d={int(row['impressions_90d']):,} -- "
        f"real demand sitting on a page that under-earns for its rank."
    )
    if row["ctr"] < 0.05:
        wrong_note = (
            f"CTR is near-zero ({row['ctr']:.2f}%) despite a page_1 position -- that's more consistent "
            f"with a technical/indexing problem (noindex, broken canonical, cannibalizing URL, GSC "
            f"attribution gap) than a content gap. A content refresh wouldn't fix a technical block; "
            f"worth a quick indexing/canonical check before assigning this as a content-fix action."
        )
    elif row["days_since_last_update"] >= 180:
        wrong_note = (
            f"stale for {int(row['days_since_last_update'])}d -- if a refresh already shipped and "
            f"CTR hasn't caught up in GSC's reporting lag yet, this pick is premature."
        )
    elif row["impressions_90d"] < 1000:
        wrong_note = (
            "demand is on the smaller side for this tier -- if the true 90d CTR benchmark for "
            "this exact query intent is naturally lower than the tier median, the 'gap' is an "
            "artifact of a coarse benchmark, not a real underperformance."
        )
    else:
        wrong_note = (
            "would be wrong if the low CTR is actually a title/meta A-B test in progress, or the "
            "SERP has a heavy answer-box/ads takeover suppressing CTR for reasons a content refresh can't fix."
        )
    print(f"    what would make it wrong: {wrong_note}")
    print()


# 1  content_id=content_c8e9d6ab9013
    action: fix_ctr_snippet_and_meta  |  reason: ctr_underperforming_visible_page
    why: position_tier=page_1 (avg_position=9.7), ctr=0.00% vs tier benchmark 0.24% (gap=0.24pp), impressions_90d=208,678 -- real demand sitting on a page that under-earns for its rank.
    what would make it wrong: CTR is near-zero (0.00%) despite a page_1 position -- that's more consistent with a technical/indexing problem (noindex, broken canonical, cannibalizing URL, GSC attribution gap) than a content gap. A content refresh wouldn't fix a technical block; worth a quick indexing/canonical check before assigning this as a content-fix action.

# 2  content_id=content_453722754fea
    action: fix_ctr_snippet_and_meta  |  reason: ctr_underperforming_visible_page
    why: position_tier=page_1 (avg_position=7.6), ctr=0.01% vs tier benchmark 0.24% (gap=0.23pp), impressions_90d=140,079 -- real demand sitting on a page that under-earns for its rank.
    what would make it w

## 4. Weak picks + leakage check

Which top-10 picks look weakest, and why -- then confirm no future-window or label-derived
columns leaked into the score.


In [4]:
# --- Weakest picks in the top 10 ---
weak = top10.copy()
# Every top-10 row has a decent position (avg_position < 10) but CTR near 0.00-0.03% --
# that's suspiciously low even for a page below its tier benchmark. A real page-1 position
# with near-zero CTR more often points at a technical/indexing/tracking problem (noindex,
# broken canonical, GSC attribution gap, cannibalization by another URL) than a content
# problem a refresh would fix -- so a content-side action could be the wrong prescription here.
weak["weakness_flag"] = np.where(
    weak["ctr"] < 0.05, "CTR near-zero despite page_1 position -- suspect technical/indexing issue, not a content gap",
    np.where(weak["days_since_last_update"] >= 180, "possibly-already-refreshed / stale flag risk",
    np.where(weak["impressions_90d"] < 1000, "smaller demand -- benchmark may be too coarse for this row", ""))
)
print(weak[["baseline_rank", "content_id", "position_tier", "ctr_gap", "impressions_90d",
            "days_since_last_update", "weakness_flag"]].to_string(index=False))

n_weak = (weak["weakness_flag"] != "").sum()
print(f"\n{n_weak} of the top 10 carry a flagged weakness -- if this were 0, I'd look harder for one,")
print("per the building-baselines skill's verification check.")

# --- Leakage check ---
SCORE_INPUT_COLUMNS = ["avg_position", "position_tier", "ctr", "impressions_90d"]
FORBIDDEN = {"trend_direction", "trend_pct", "is_declining_label"}
leaked = FORBIDDEN.intersection(SCORE_INPUT_COLUMNS)
print(f"\nColumns that built the score: {SCORE_INPUT_COLUMNS}")
print(f"Forbidden (label-derived) columns: {sorted(FORBIDDEN)}")
print(f"Overlap (should be empty): {sorted(leaked)}")
assert not leaked, "Leakage: a label-derived column was used to build the score."

FUTURE_WINDOW_COLUMNS = ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                          "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
future_leak = set(FUTURE_WINDOW_COLUMNS).intersection(SCORE_INPUT_COLUMNS)
print(f"30-day comparison-window columns used in the score (should be empty): {sorted(future_leak)}")
assert not future_leak, "Leakage: a trend-comparison-window column was used to build the score."

print("\nNo future-window or label-derived inputs went into baseline_action_score. Clean.")

print("\nTakeaway: ALL 10 top picks share the same weakness -- the score's highest ranks are")
print("dominated by near-zero-CTR pages, because ctr_gap is largest exactly when ctr sits at ~0.")
print("That's an honest flaw in this baseline, not just a one-off bad row: it conflates 'CTR fix")
print("opportunity' with 'possible technical/indexing break'. A stronger v2 would either exclude")
print("near-zero-CTR rows from this rule (route them to a technical-audit queue instead) or cap")
print("ctr_gap so it can't be maximized purely by ctr approaching 0. Leaving it as-is here --")
print("this is exactly the kind of gap the Week-5 model should be able to close.")


 baseline_rank           content_id position_tier  ctr_gap  impressions_90d  days_since_last_update                                                                                weakness_flag
             1 content_c8e9d6ab9013        page_1     0.24           208678                     104 CTR near-zero despite page_1 position -- suspect technical/indexing issue, not a content gap
             2 content_453722754fea        page_1     0.23           140079                      20 CTR near-zero despite page_1 position -- suspect technical/indexing issue, not a content gap
             3 content_39881853ef0c        page_1     0.23           112434                      20 CTR near-zero despite page_1 position -- suspect technical/indexing issue, not a content gap
             4 content_c84a0ab98e90        page_1     0.21           223271                      20 CTR near-zero despite page_1 position -- suspect technical/indexing issue, not a content gap
             5 content_0919dd345d80

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.